# ML-07 - Baseline Action Score and Top-10 Review

**Lane:** Refresh / Content Opportunity Scoring

This notebook creates a transparent, rule-based queue for human content review. It is a baseline for a later model to beat, not an automated instruction to change pages.

## 1. My rule and its reason code

### Rule in plain words

Recommend a page for **content-refresh review** when it has not been updated for at least 90 days and still receives at least 300 search impressions in the trailing 90-day snapshot. Rank eligible pages by a capped freshness weight multiplied by `log1p(impressions_90d)`. The log keeps a few exceptionally large pages from completely dominating the queue, and the 180-day cap reflects the signal check below: the relationship is not shown to keep strengthening forever.

The queue has one reason code: `stale_visible_content`. Its action label is **Review for content refresh**.

The score uses only `days_since_last_update` and `impressions_90d`, which are snapshot-time inputs. `trend_direction`, `trend_pct`, and all 30-day comparison-window fields are excluded from the rule. The decline proxy below is used only to describe the data and evaluate this baseline after scoring; it is not an input.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "raw" / "content_refresh_anonymized.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
pd.set_option("display.max_colwidth", 120)

# Signal check 1: freshness is behind FlyRank's refresh-style flags.
df["is_declining_proxy"] = (df["trend_direction"] == "down").astype(int)
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 30, 90, 180, 365, np.inf],
    labels=["0-30", "31-90", "91-180", "181-365", "365+"],
    include_lowest=True,
)

staleness_check = (
    df.groupby("staleness_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        median_impressions_90d=("impressions_90d", "median"),
        declining_proxy_share=("is_declining_proxy", "mean"),
    )
    .reset_index()
)
staleness_check["declining_proxy_share"] = staleness_check["declining_proxy_share"].round(3)
print("Signal 1 - staleness (flag-linked): MIXED")
print("The 91-180 day bucket has a higher observed decline-proxy share than 0-30 days, but the pattern is not monotonic. The 365+ bucket has n < 50, so it is not used for a verdict.")
display(staleness_check)

# Signal check 2: visibility is behind quick-win logic, but it is an impact screen rather than a decline predictor.
impression_check = (
    df.groupby("impression_tier", observed=True)
    .agg(
        n=("content_id", "size"),
        median_impressions_90d=("impressions_90d", "median"),
        declining_proxy_share=("is_declining_proxy", "mean"),
    )
    .reset_index()
)
impression_check["declining_proxy_share"] = impression_check["declining_proxy_share"].round(3)
print("\nSignal 2 - impressions / visibility: MIXED")
print("Middle visibility tiers have the largest observed decline-proxy shares; excellent visibility does not. I use impressions to prioritize potential impact, not as a claim that more impressions cause decline.")
display(impression_check)


Signal 1 - staleness (flag-linked): MIXED
The 91-180 day bucket has a higher observed decline-proxy share than 0-30 days, but the pattern is not monotonic. The 365+ bucket has n < 50, so it is not used for a verdict.

Signal 2 - impressions / visibility: MIXED
Middle visibility tiers have the largest observed decline-proxy shares; excellent visibility does not. I use impressions to prioritize potential impact, not as a claim that more impressions cause decline.


,staleness_bucket,n,median_impressions_90d,declining_proxy_share
0,0-30,20480,470.0,0.511
1,31-90,175,510.0,0.589
2,91-180,9171,1692.0,0.611
3,181-365,169,16.0,0.467
4,365+,5,2.0,0.600


,impression_tier,n,median_impressions_90d,declining_proxy_share
0,excellent,1078,48675.0,0.462
1,good,7205,7249.0,0.586
2,low,11248,31.0,0.454
3,moderate,10469,998.0,0.615


None

## 2. Build the ranked queue (writes the CSV)

The score is deliberately simple and reproducible:

`eligible = updated 90+ days ago AND impressions_90d >= 300`

`baseline_score = eligible * min(days_since_last_update, 180) / 90 * log1p(impressions_90d)`

This is decision support: it places potentially consequential refresh reviews near the top. It does not claim that every selected page is declining or that refreshing it will improve performance.

In [2]:
# The score inputs are listed explicitly to make the leakage boundary easy to inspect.
score_inputs = ["days_since_last_update", "impressions_90d"]
min_days_since_update = 90
max_days_weighted = 180
min_impressions = 300

eligible = (
    df["days_since_last_update"].ge(min_days_since_update)
    & df["impressions_90d"].ge(min_impressions)
)
freshness_weight = df["days_since_last_update"].clip(upper=max_days_weighted) / min_days_since_update

df["baseline_score"] = np.where(
    eligible,
    freshness_weight * np.log1p(df["impressions_90d"]),
    0.0,
)
df["reason_code"] = np.where(eligible, "stale_visible_content", "not_eligible")
df["action_label"] = np.where(eligible, "Review for content refresh", "No baseline action")

queue_columns = [
    "content_id", "client_id", "baseline_score", "reason_code", "action_label",
    "days_since_last_update", "freshness_tier", "impressions_90d", "clicks_90d",
    "avg_position", "position_tier",
]
queue = (
    df.loc[eligible, queue_columns]
    .sort_values(["baseline_score", "content_id"], ascending=[False, True])
    .reset_index(drop=True)
)

output_dir = ROOT / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
queue_path = output_dir / "baseline_action_score.csv"
queue.to_csv(queue_path, index=False)

# The proxy is an evaluation receipt only. It was not used in the score or CSV.
evaluation_order = df.loc[eligible].sort_values(["baseline_score", "content_id"], ascending=[False, True])
precision_at_50 = float(evaluation_order.head(50)["is_declining_proxy"].mean())
base_rate = float(df["is_declining_proxy"].mean())
metrics = {
    "lane": "Refresh / Content Opportunity Scoring",
    "rows_scored": int(len(df)),
    "eligible_queue_rows": int(len(queue)),
    "score_inputs": score_inputs,
    "thresholds": {"min_days_since_last_update": min_days_since_update, "min_impressions_90d": min_impressions, "max_days_weighted": max_days_weighted},
    "diagnostic_only_declining_proxy_base_rate": round(base_rate, 4),
    "diagnostic_only_precision_at_50": round(precision_at_50, 4),
}
with (output_dir / "baseline_action_score_metrics.json").open("w", encoding="utf-8") as handle:
    json.dump(metrics, handle, indent=2)

print(f"Wrote {len(queue):,} ranked rows to {queue_path.relative_to(ROOT)}")
print(f"Diagnostic only: precision@50 = {precision_at_50:.3f}; full-data decline-proxy base rate = {base_rate:.3f}")
print("The difference is not a causal result and should not be used to alter the rule after seeing this proxy.")
display(queue.head(20))


Wrote 7,234 ranked rows to work\outputs\baseline_action_score.csv
Diagnostic only: precision@50 = 0.560; full-data decline-proxy base rate = 0.542
The difference is not a causal result and should not be used to alter the rule after seeing this proxy.


,content_id,client_id,baseline_score,reason_code,action_label,days_since_last_update,freshness_tier,impressions_90d,clicks_90d,avg_position,position_tier
0,content_cf56e2e2e282,client_7f2253d7e2,22.059398,stale_visible_content,Review for content refresh,194,181+,61678,94,19.7,striking
1,content_7368877ea310,client_7f2253d7e2,21.986555,stale_visible_content,Review for content refresh,194,181+,59472,77,24.8,page_3_5
2,content_1bfaa38ff26c,client_7f2253d7e2,20.309737,stale_visible_content,Review for content refresh,194,181+,25715,60,22.2,page_3_5
3,content_0a91db491d14,client_7f2253d7e2,18.991039,stale_visible_content,Review for content refresh,193,181+,13299,65,10.5,striking
4,content_5feee3994adb,client_7f2253d7e2,17.927089,stale_visible_content,Review for content refresh,194,181+,7812,1,39.0,page_3_5
5,content_c2d929d83eaa,client_7f2253d7e2,17.860988,stale_visible_content,Review for content refresh,193,181+,7558,15,17.9,striking
6,content_b16bd7307b39,client_7f2253d7e2,16.863706,stale_visible_content,Review for content refresh,194,181+,4590,0,31.0,page_3_5
7,content_fe16a55cd13d,client_7f2253d7e2,16.848840,stale_visible_content,Review for content refresh,194,181+,4556,15,16.4,striking
8,content_ecb6215e79fd,client_7f2253d7e2,16.792310,stale_visible_content,Review for content refresh,194,181+,4429,17,25.3,page_3_5
9,content_cb7e312f5d32,client_9f14025af0,16.719381,stale_visible_content,Review for content refresh,151,91-180,21272,522,12.6,striking


None

## 3. Top-10 review

Each recommendation below is a request for human review. The notes name the page-level evidence in the score and a realistic condition that could make the suggestion wrong.

In [3]:
def why_wrong(row):
    if row["clicks_90d"] == 0:
        return "It could be wrong if zero clicks reflect tracking or query-matching limits, or if search demand is too weak for a content refresh to help."
    if row["avg_position"] == 0:
        return "It could be wrong if position data are unavailable; this rule does not establish that a refresh can improve search visibility."
    if row["avg_position"] <= 10:
        return "It could be wrong if the page already ranks well and is intentionally stable; a refresh could add work without meaningful upside."
    return "It could be wrong if the weak search position is caused by a technical, indexing, seasonal, or strategic issue rather than stale content."

top10_review = queue.head(10).copy()
top10_review["why_it_is_here"] = top10_review.apply(
    lambda row: f'{int(row["days_since_last_update"])} days since update and {int(row["impressions_90d"]):,} trailing-90-day impressions.',
    axis=1,
)
top10_review["confidence_note"] = "Medium: a transparent screen for review priority, not a guaranteed uplift."
top10_review["what_would_make_it_wrong"] = top10_review.apply(why_wrong, axis=1)
review_columns = [
    "content_id", "action_label", "reason_code", "why_it_is_here",
    "confidence_note", "what_would_make_it_wrong",
]
display(top10_review[review_columns])


,content_id,action_label,reason_code,why_it_is_here,confidence_note,what_would_make_it_wrong
0,content_cf56e2e2e282,Review for content refresh,stale_visible_content,"194 days since update and 61,678 trailing-90-day impressions.","Medium: a transparent screen for review priority, not a guaranteed uplift.","It could be wrong if the weak search position is caused by a technical, indexing, seasonal, or strategic issue rather than stale content."
1,content_7368877ea310,Review for content refresh,stale_visible_content,"194 days since update and 59,472 trailing-90-day impressions.","Medium: a transparent screen for review priority, not a guaranteed uplift.","It could be wrong if the weak search position is caused by a technical, indexing, seasonal, or strategic issue rather than stale content."
2,content_1bfaa38ff26c,Review for content refresh,stale_visible_content,"194 days since update and 25,715 trailing-90-day impressions.","Medium: a transparent screen for review priority, not a guaranteed uplift.","It could be wrong if the weak search position is caused by a technical, indexing, seasonal, or strategic issue rather than stale content."
3,content_0a91db491d14,Review for content refresh,stale_visible_content,"193 days since update and 13,299 trailing-90-day impressions.","Medium: a transparent screen for review priority, not a guaranteed uplift.","It could be wrong if the weak search position is caused by a technical, indexing, seasonal, or strategic issue rather than stale content."
4,content_5feee3994adb,Review for content refresh,stale_visible_content,"194 days since update and 7,812 trailing-90-day impressions.","Medium: a transparent screen for review priority, not a guaranteed uplift.","It could be wrong if the weak search position is caused by a technical, indexing, seasonal, or strategic issue rather than stale content."
5,content_c2d929d83eaa,Review for content refresh,stale_visible_content,"193 days since update and 7,558 trailing-90-day impressions.","Medium: a transparent screen for review priority, not a guaranteed uplift.","It could be wrong if the weak search position is caused by a technical, indexing, seasonal, or strategic issue rather than stale content."
6,content_b16bd7307b39,Review for content refresh,stale_visible_content,"194 days since update and 4,590 trailing-90-day impressions.","Medium: a transparent screen for review priority, not a guaranteed uplift.","It could be wrong if zero clicks reflect tracking or query-matching limits, or if search demand is too weak for a content refresh to help."
7,content_fe16a55cd13d,Review for content refresh,stale_visible_content,"194 days since update and 4,556 trailing-90-day impressions.","Medium: a transparent screen for review priority, not a guaranteed uplift.","It could be wrong if the weak search position is caused by a technical, indexing, seasonal, or strategic issue rather than stale content."
8,content_ecb6215e79fd,Review for content refresh,stale_visible_content,"194 days since update and 4,429 trailing-90-day impressions.","Medium: a transparent screen for review priority, not a guaranteed uplift.","It could be wrong if the weak search position is caused by a technical, indexing, seasonal, or strategic issue rather than stale content."
9,content_cb7e312f5d32,Review for content refresh,stale_visible_content,"151 days since update and 21,272 trailing-90-day impressions.","Medium: a transparent screen for review priority, not a guaranteed uplift.","It could be wrong if the weak search position is caused by a technical, indexing, seasonal, or strategic issue rather than stale content."


None

## 4. Weak picks and leakage check

A rule should reveal its own blind spots. The examples below are eligible pages that deserve extra caution because the rule can value age and impressions even when another explanation may be more important.

Leakage check: the score is built from `days_since_last_update` and `impressions_90d` only. It does not use `trend_direction`, `trend_pct`, `is_declining_proxy`, IDs, or any `*_last_30d` / `*_prev_30d` field.

In [4]:
weak_pick_mask = (queue["clicks_90d"].eq(0)) | (queue["impressions_90d"].lt(500))
weak_picks = queue.loc[weak_pick_mask].head(10).copy()
weak_picks["why_caution_is_needed"] = np.where(
    weak_picks["clicks_90d"].eq(0),
    "No recorded clicks: content may not be the main constraint, or the measurement may be incomplete.",
    "Near the visibility threshold: a high score can be driven mainly by age rather than broad demand.",
)

forbidden_score_inputs = [
    "trend_direction", "trend_pct", "is_declining_proxy", "content_id", "client_id",
    "impressions_last_30d", "impressions_prev_30d", "clicks_last_30d", "clicks_prev_30d",
]
assert not set(score_inputs).intersection(forbidden_score_inputs)
assert not any("last_30d" in column or "prev_30d" in column for column in score_inputs)

print("Leakage check passed: score inputs are", score_inputs)
print("Caution examples (these are not automatically bad pages):")
display(weak_picks[["content_id", "baseline_score", "days_since_last_update", "impressions_90d", "clicks_90d", "why_caution_is_needed"]])


Leakage check passed: score inputs are ['days_since_last_update', 'impressions_90d']
Caution examples (these are not automatically bad pages):


,content_id,baseline_score,days_since_last_update,impressions_90d,clicks_90d,why_caution_is_needed
6,content_b16bd7307b39,16.863706,194,4590,0,"No recorded clicks: content may not be the main constraint, or the measurement may be incomplete."
22,content_c8e9d6ab9013,14.153883,104,208678,0,"No recorded clicks: content may not be the main constraint, or the measurement may be incomplete."
249,content_074ba6ead17b,12.560792,183,533,0,"No recorded clicks: content may not be the main constraint, or the measurement may be incomplete."
398,content_fd16e3475c29,12.127570,183,429,0,"No recorded clicks: content may not be the main constraint, or the measurement may be incomplete."
559,content_b65fe2792b44,11.837788,183,371,2,Near the visibility threshold: a high score can be driven mainly by age rather than broad demand.
645,content_ba00ffc6318c,11.692878,211,345,17,Near the visibility threshold: a high score can be driven mainly by age rather than broad demand.
646,content_df71843dcd17,11.691557,103,27334,0,"No recorded clicks: content may not be the main constraint, or the measurement may be incomplete."
674,content_4729b57ca036,11.634222,301,335,11,Near the visibility threshold: a high score can be driven mainly by age rather than broad demand.
837,content_6476d1d8c050,11.440624,313,304,0,"No recorded clicks: content may not be the main constraint, or the measurement may be incomplete."
1021,content_825a9788af8d,11.241661,104,16786,0,"No recorded clicks: content may not be the main constraint, or the measurement may be incomplete."


None

## Self-check

- [x] Every section includes both reasoning and code.
- [x] Two signal bucket tables show visible `n` values; staleness is linked to FlyRank refresh logic.
- [x] One transparent score, one reason code, and one action label are written to a ranked CSV.
- [x] The top ten include an action, reason, confidence note, and a condition that could make the recommendation wrong.
- [x] The score excludes future windows, label-derived fields, and IDs.
- [x] Claims are observational and decision-support oriented.